*****Imports and Configurations*****

In [49]:
import json
import os
from pathlib import Path
import numpy as np
import easyocr
import pypdfium2 as pdfium
import chromadb
from chromadb.utils import embedding_functions

PDF_DIR = Path(r"C:\Users\aline\PycharmProjects\EducAgent\Educagent\Math Curriculum")
CACHE_DIR = PDF_DIR.parent / "ocr_cache"
DB_PATH = str(PDF_DIR.parent / "chroma_db")

CACHE_DIR.mkdir(exist_ok=True)
PDF_PATHS = sorted(PDF_DIR.glob("*.pdf"))

for p in PDF_PATHS:
    print(p.name)
print(len(PDF_PATHS), "PDFs found")

Math_EN_P4_T1.pdf
Math_EN_P6_T1.pdf
Math_EN_prim5_t1.pdf
3 PDFs found


*****OCR*****

In [50]:
reader = None

def get_reader():
    global reader
    if reader is None:
        reader = easyocr.Reader(["en"])
    return reader

def page_text(page):
    native = page.get_textpage().get_text_range().strip()
    if len(native) > 100:
        return native
    img = np.array(page.render(scale=150 / 72).to_pil().convert("RGB"))
    return " ".join(get_reader().readtext(img, detail=0, paragraph=True))

def load_book(pdf_path):
    cache = CACHE_DIR / f"{pdf_path.stem}.json"
    pages = json.loads(cache.read_text(encoding="utf-8")) if cache.exists() else []
    pdf = pdfium.PdfDocument(str(pdf_path))
    total = len(pdf)
    for i in range(len(pages), total):
        pages.append([i + 1, page_text(pdf[i])])
        if (i + 1) % 10 == 0 or i + 1 == total:
            cache.write_text(json.dumps(pages, ensure_ascii=False), encoding="utf-8")
            print(f"{pdf_path.name}: {i + 1}/{total}")
    return [tuple(p) for p in pages]

books = {p.stem: load_book(p) for p in PDF_PATHS}

for name, pages in books.items():
    print(name, len(pages), sum(len(t) for _, t in pages))

Math_EN_P4_T1 136 116717
Math_EN_P6_T1 122 132500
Math_EN_prim5_t1 140 118777


*****Chunking*****

In [51]:
import re

def grade_of(name):
    m = re.search(r"P(\d)|prim(\d)", name, re.IGNORECASE)
    return int(next(g for g in m.groups() if g)) if m else None

def chunk_book(name, pages, size=150, overlap=30):
    out = []
    step = size - overlap
    for page_no, text in pages:
        words = text.split()
        for i in range(0, max(len(words), 1), step):
            piece = " ".join(words[i:i + size])
            if piece.strip():
                out.append({"source": name, "grade": grade_of(name), "page": page_no, "text": piece})
    return out

def chunk_all(books, size=150, overlap=30):
    chunks = []
    for name, pages in books.items():
        chunks.extend(chunk_book(name, pages, size, overlap))
    return chunks

chunks = chunk_all(books)
print(len(chunks))
for name in books:
    print(name, grade_of(name), sum(c["source"] == name for c in chunks))

735
Math_EN_P4_T1 4 240
Math_EN_P6_T1 6 253
Math_EN_prim5_t1 5 242


*****Embeddings and Chroma*****

In [52]:
client = chromadb.PersistentClient(path=DB_PATH)
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

def build_collection(name, size=150, overlap=30):
    try:
        client.delete_collection(name)
    except Exception:
        pass
    col = client.create_collection(
        name=name,
        embedding_function=embed_fn,
        metadata={"hnsw:space": "cosine"},
    )
    items = chunk_all(books, size, overlap)
    for start in range(0, len(items), 500):
        batch = items[start:start + 500]
        col.add(
            ids=[f"{c['source']}_p{c['page']}_{start + j}" for j, c in enumerate(batch)],
            documents=[c["text"] for c in batch],
            metadatas=[{"source": c["source"], "grade": c["grade"] or 0, "page": c["page"]} for c in batch],
        )
    return col

collection = build_collection("math_curriculum")
print(collection.count())

735


*****Search with a grade filter*****

In [53]:
def search(q, col=None, k=4, grade=None, show=True):
    col = col or collection
    r = col.query(
        query_texts=[q],
        n_results=k,
        where={"grade": grade} if grade else None,
    )
    hits = [
        (m["source"], m["page"], 1 - d, doc)
        for doc, m, d in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])
    ]
    if show:
        for source, page, sim, doc in hits:
            print(f"{source}  page {page}  cosine {sim:.3f}")
            print(doc[:300], "\n")
    return hits

search("What is a bar model?")
search("What is a bar model?", grade=4)

Math_EN_P6_T1  page 95  cosine 0.411
let's explore some other types of graphs Bar Graphs and Histograms Suppose Amira's class constructed these bar graphs to show some other data they collected from different groups of people about their movie-watching habits. Favorite Cinema Favorite Actors 25 20 1 15 6 10 L West Central East Cinema C 

Math_EN_P4_T1  page 53  cosine 0.393
the following Conversion Example: 5,403 mL (1,000 mL=1L) 5,000 mL = 5L 403 mL 5 L 403 mL Using the conversion examples above, convert the following weights into the given units on the bar models_ 6. 9,425 mL 6,360 mL mL mL Lesson 3: Units of Capacity 45 

Math_EN_P4_T1  page 40  cosine 0.385
UNIT 2 CONCEPT 2 Solving Multistep Problems LESSON 4 Bar Models, Variables, and Story Problems Learning Targets can use letters to represent unknown quantities in equations_ can use bar models to represent and solve story problems_ can solve equations that include variables_ ACCESS Introducing Varia 

Math_EN_P4_T1  page 42  cos

[('Math_EN_P4_T1',
  53,
  0.3926602005958557,
  'the following Conversion Example: 5,403 mL (1,000 mL=1L) 5,000 mL = 5L 403 mL 5 L 403 mL Using the conversion examples above, convert the following weights into the given units on the bar models_ 6. 9,425 mL 6,360 mL mL mL Lesson 3: Units of Capacity 45'),
 ('Math_EN_P4_T1',
  40,
  0.3849674463272095,
  'UNIT 2 CONCEPT 2 Solving Multistep Problems LESSON 4 Bar Models, Variables, and Story Problems Learning Targets can use letters to represent unknown quantities in equations_ can use bar models to represent and solve story problems_ can solve equations that include variables_ ACCESS Introducing Variables What does it mean to be balanced? Look at the image of the triangle: What do you notice about it? Discuss your observations with your Shoulder Partner: BUILD Bar Models Read the following problems. Create a bar model and an equation for each problem; and then solve: There are 5,328 ants in the colony: In the colony, 2,164 ants are femal

*****Test set*****

In [54]:
test_set = [
    ("How do I solve the ants colony problem with a bar model?", "Math_EN_P4_T1", [40]),
    ("How do I round a number to the nearest ten thousand?", "Math_EN_P4_T1", [28, 29, 30]),
    ("What is the commutative property of multiplication?", "Math_EN_P4_T1", [85, 86]),
    ("How do I find the perimeter of a rectangle?", "Math_EN_P4_T1", [67, 68, 69]),
    ("What is the difference between prime and composite numbers?", "Math_EN_P4_T1", [96, 97, 98]),
    ("How do I find the greatest common factor of two numbers?", "Math_EN_P4_T1", [99, 100]),
    ("How do I calculate elapsed time?", "Math_EN_P4_T1", [58, 59]),
    ("How do I use the standard division algorithm?", "Math_EN_P4_T1", [127, 128, 129]),
    ("Which lesson covers the partial products algorithm?", "Math_EN_P4_T1", [112, 113, 114]),
    ("How do I convert between units of capacity?", "Math_EN_P4_T1", [53, 54, 55]),
    ("How much time passed between 3:15 and 5:40?", "Math_EN_P4_T1", [58, 59]),

    ("How do I find the prime factorization of a number?", "Math_EN_prim5_t1", [59, 60]),
    ("How do I find the least common multiple?", "Math_EN_prim5_t1", [67, 69]),
    ("How do I multiply a decimal by a power of ten?", "Math_EN_prim5_t1", [100, 101, 102]),
    ("How do I use the area model to multiply?", "Math_EN_prim5_t1", [73, 74, 75]),
    ("How do I model decimal addition?", "Math_EN_prim5_t1", [30, 31, 32]),
    ("How do I divide decimals by decimals?", "Math_EN_prim5_t1", [126, 127]),
    ("How do parentheses change the value of a numerical expression?", "Math_EN_prim5_t1", [131, 132, 133]),

    ("How do I write algebraic expressions?", "Math_EN_P6_T1", [52, 53, 54]),
    ("What are the divisibility rules?", "Math_EN_P6_T1", [10, 11]),
    ("What does absolute value mean?", "Math_EN_P6_T1", [37, 38, 39]),
    ("How do I find the median, mode, and outliers?", "Math_EN_P6_T1", [114, 115, 116]),
    ("How do I find the middle value of a data set?", "Math_EN_P6_T1", [114, 115, 116]),
    ("How do I draw a histogram?", "Math_EN_P6_T1", [94, 95, 96, 97, 98]),
    ("How do I solve inequalities?", "Math_EN_P6_T1", [70, 73, 74, 75]),
    ("What is the difference between dependent and independent variables?", "Math_EN_P6_T1", [78, 80]),
    ("How do I read a box plot?", "Math_EN_P6_T1", [101, 102]),
    ("How do I evaluate an algebraic expression?", "Math_EN_P6_T1", [59, 60]),
    ("What is an exponent?", "Math_EN_P6_T1", [56, 57, 58]),
]

print(len(test_set))

29


*****Evaluation function (hit rate and MRR)*****

In [55]:
def evaluate(col, k=4, filtered=False, verbose=False):
    hits, rr = 0, 0.0
    for q, source, pages_ok in test_set:
        grade = grade_of(source) if filtered else None
        got = [(s, p) for s, p, _, _ in search(q, col, k, grade, show=False)]
        rank = next((i + 1 for i, (s, p) in enumerate(got) if s == source and p in pages_ok), None)
        hits += rank is not None
        rr += 1 / rank if rank else 0
        if verbose:
            print(rank or "MISS", "|", q)
            if rank is None or rank > 2:
                print("   expected", source, pages_ok)
                print("   got     ", got)
    return hits / len(test_set), rr / len(test_set)

*****Chunk size sweep*****

In [56]:
for size, overlap in [(100, 20), (150, 30), (250, 50), (400, 80)]:
    col = build_collection(f"sweep_{size}", size, overlap)
    for filtered in (False, True):
        hr, mrr = evaluate(col, filtered=filtered)
        print(f"size {size:>3}  chunks {col.count():>4}  filter {'on ' if filtered else 'off'}  hit@4 {hr:.2f}  MRR {mrr:.2f}")

size 100  chunks 1000  filter off  hit@4 0.93  MRR 0.85
size 100  chunks 1000  filter on   hit@4 0.93  MRR 0.88
size 150  chunks  735  filter off  hit@4 0.93  MRR 0.80
size 150  chunks  735  filter on   hit@4 0.93  MRR 0.86
size 250  chunks  484  filter off  hit@4 0.90  MRR 0.75
size 250  chunks  484  filter on   hit@4 0.93  MRR 0.80
size 400  chunks  407  filter off  hit@4 0.90  MRR 0.78
size 400  chunks  407  filter on   hit@4 0.93  MRR 0.82


*****Final collection and Weak spots*****

In [57]:
collection = build_collection("final", 100, 20)
print(collection.count())
evaluate(collection, filtered=True, verbose=True)

1000
1 | How do I solve the ants colony problem with a bar model?
1 | How do I round a number to the nearest ten thousand?
1 | What is the commutative property of multiplication?
1 | How do I find the perimeter of a rectangle?
1 | What is the difference between prime and composite numbers?
1 | How do I find the greatest common factor of two numbers?
1 | How do I calculate elapsed time?
2 | How do I use the standard division algorithm?
2 | Which lesson covers the partial products algorithm?
1 | How do I convert between units of capacity?
2 | How much time passed between 3:15 and 5:40?
1 | How do I find the prime factorization of a number?
1 | How do I find the least common multiple?
MISS | How do I multiply a decimal by a power of ten?
   expected Math_EN_prim5_t1 [100, 101, 102]
   got      [('Math_EN_prim5_t1', 122), ('Math_EN_prim5_t1', 106), ('Math_EN_prim5_t1', 116), ('Math_EN_prim5_t1', 16)]
1 | How do I use the area model to multiply?
1 | How do I model decimal addition?
1 | How 

(0.9310344827586207, 0.8793103448275862)

*****Page-level retrieval for the LLM*****

In [58]:
page_lookup = {(name, p): t for name, pages in books.items() for p, t in pages}

def retrieve_pages(q, grade=None, n_pages=3, k=12):
    seen, out = set(), []
    for source, page, sim, _ in search(q, collection, k, grade, show=False):
        if (source, page) not in seen:
            seen.add((source, page))
            out.append({"source": source, "page": page, "similarity": sim, "text": page_lookup[(source, page)]})
        if len(out) == n_pages:
            break
    return out

for r in retrieve_pages("How do I calculate elapsed time?", grade=4):
    print(r["source"], r["page"], round(r["similarity"], 3), len(r["text"].split()), "words")

Math_EN_P4_T1 59 0.529 200 words
Math_EN_P4_T1 57 0.523 279 words
Math_EN_P4_T1 60 0.496 211 words


*****Install SDK*****

In [59]:
import sys
!{sys.executable} -m pip install openai python-dotenv

You should consider upgrading via the 'C:\Users\aline\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip' command.


*****LLM setup (Gemini 3.5 Flash)*****

In [60]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))

gemini_key = os.environ.get("GEMINI_API_KEY", "").strip()
openrouter_key = os.environ.get("OPENROUTER_API_KEY", "").strip()

# Gemini 3.x models "think" before answering, and those hidden thinking tokens count
# against max_tokens. With a small budget (e.g. 800) the thinking used it all up and the
# visible answer came back truncated or empty. So: give a generous budget, keep
# reasoning low, and treat a cut-off / empty answer as a failure instead of returning it.
MAX_TOKENS = 4096

PROVIDERS = []
if gemini_key:
    PROVIDERS.append((
        "Gemini",
        OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
               api_key=gemini_key, timeout=60, max_retries=1),
        ["gemini-3.5-flash", "gemini-3.8-flash", "gemini-flash-latest"],
        {"reasoning_effort": "low"},
    ))
if openrouter_key:
    PROVIDERS.append((
        "OpenRouter",
        OpenAI(base_url="https://openrouter.ai/api/v1",
               api_key=openrouter_key, timeout=60, max_retries=1),
        ["meta-llama/llama-3.3-70b-instruct", "deepseek/deepseek-chat"],
        {},
    ))

def generate(system, user):
    if not PROVIDERS:
        raise RuntimeError("No API key found. Put GEMINI_API_KEY and/or OPENROUTER_API_KEY in Educagent/.env")
    errors = []
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    for provider, client, models, extra in PROVIDERS:
        for m in models:
            try:
                r = client.chat.completions.create(model=m, max_tokens=MAX_TOKENS, messages=messages, **extra)
                choice = r.choices[0]
                text = (choice.message.content or "").strip()
                if not text:
                    raise ValueError(f"empty answer (finish_reason={choice.finish_reason})")
                if choice.finish_reason == "length":
                    raise ValueError("answer was cut off (finish_reason=length)")
                return text
            except Exception as e:
                errors.append(f"{provider} {m}: {e}")
    raise RuntimeError("Generation failed:\n" + "\n".join(errors))


*****Test*****

In [61]:
SYSTEM = (
    "You are a math tutor for primary school students. "
    "Answer only from the textbook pages provided, in simple language suited to the student's grade. "
    "Cite the pages you used like (p. 40). "
    "The pages come from OCR and may contain small errors, so fix obvious typos silently. "
    "If the pages do not contain the answer, say so instead of guessing."
)

def ask(question, grade=None, n_pages=3, show_sources=True):
    try:
        pages = retrieve_pages(question, grade, n_pages)
        context = "\n\n".join(f"[{p['source']} p.{p['page']}]\n{p['text']}" for p in pages)
    except Exception as e:
        context = f"(Note: Retrieval skipped: {e})"
        pages = []
    user = f"Student grade: {grade or 'unknown'}\n\nTextbook pages:\n{context}\n\nQuestion: {question}"
    if show_sources and pages:
        print("Retrieved:", [(p["source"], p["page"], round(p["similarity"], 2)) for p in pages], "\n")
    answer = generate(SYSTEM, user)
    print(answer)
    return answer

In [62]:
question = "How do I solve the ants colony problem with a bar model?"
grade = 4

ask(question, grade)

Retrieved: [('Math_EN_P4_T1', 40, 0.76), ('Math_EN_P4_T1', 41, 0.67), ('Math_EN_P4_T1', 45, 0.54)] 

To solve an ant colony story problem using a bar model, follow these steps:

1. **Find the Total (the Whole):** Look for the total number of ants in the colony. Put this total number in the top bar of your model (p. 40).
2. **Find the Known Part:** Look for the group of ants you already know (like the number of female ants or ants taking out trash). Put this number in one of the bottom boxes (p. 40, p. 41).
3. **Use a Variable for the Unknown Part:** Put a letter (variable) or question mark in the other bottom box to represent the ants you need to find (like male ants or foraging ants) (p. 40, p. 41).
4. **Write an Equation and Solve:** Subtract the known part from the total to find your answer (p. 40, p. 41).

---

### **Example from the Book:**
* **Problem:** There are 5,328 ants in a colony. 2,164 are females, and the rest are males. How many male ants are there? (p. 40)

* **Bar Mod

'To solve an ant colony story problem using a bar model, follow these steps:\n\n1. **Find the Total (the Whole):** Look for the total number of ants in the colony. Put this total number in the top bar of your model (p. 40).\n2. **Find the Known Part:** Look for the group of ants you already know (like the number of female ants or ants taking out trash). Put this number in one of the bottom boxes (p. 40, p. 41).\n3. **Use a Variable for the Unknown Part:** Put a letter (variable) or question mark in the other bottom box to represent the ants you need to find (like male ants or foraging ants) (p. 40, p. 41).\n4. **Write an Equation and Solve:** Subtract the known part from the total to find your answer (p. 40, p. 41).\n\n---\n\n### **Example from the Book:**\n* **Problem:** There are 5,328 ants in a colony. 2,164 are females, and the rest are males. How many male ants are there? (p. 40)\n\n* **Bar Model:**\n  * **Top Box (Total):** 5,328\n  * **Bottom Box 1 (Known Part):** 2,164\n  * **B